<a href="https://colab.research.google.com/github/MalezhykDaniil/ML/blob/main/%D0%9B%D0%A0_11_%D0%9C%D0%9D_%D0%9C%D0%B0%D0%BB%D0%B5%D0%B6%D0%B8%D0%BA_3_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
meowmeowmeowmeowmeow_gtsrb_german_traffic_sign_path = kagglehub.dataset_download('meowmeowmeowmeowmeow/gtsrb-german-traffic-sign')

print('Data source import complete.')


Using Colab cache for faster access to the 'gtsrb-german-traffic-sign' dataset.
Data source import complete.


In [2]:
import tensorflow as tf
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import Sequential

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os

from tensorflow.keras.layers import (Input, Rescaling, RandomFlip, RandomRotation, RandomContrast,
                                     Conv2D, BatchNormalization, MaxPool2D,
                                     GlobalAveragePooling2D, Dense, Dropout)

from tensorflow.keras.regularizers import l2

In [3]:
train_csv_path = "/kaggle/input/gtsrb-german-traffic-sign/Train.csv"
test_csv_path = "/kaggle/input/gtsrb-german-traffic-sign/Test.csv"
base_path = "/kaggle/input/gtsrb-german-traffic-sign"


train_df = pd.read_csv(train_csv_path)
test_df = pd.read_csv(test_csv_path)

train_df.head()

,Width,Height,Roi.X1,Roi.Y1,Roi.X2,Roi.Y2,ClassId,Path
0,27,26,5,5,22,20,20,Train/20/00020_00000_00000.png
1,28,27,5,6,23,22,20,Train/20/00020_00000_00001.png
2,29,26,6,5,24,21,20,Train/20/00020_00000_00002.png
3,28,27,5,6,23,22,20,Train/20/00020_00000_00003.png
4,28,26,5,5,23,21,20,Train/20/00020_00000_00004.png


In [4]:
test_df.head()

,Width,Height,Roi.X1,Roi.Y1,Roi.X2,Roi.Y2,ClassId,Path
0,53,54,6,5,48,49,16,Test/00000.png
1,42,45,5,5,36,40,1,Test/00001.png
2,48,52,6,6,43,47,38,Test/00002.png
3,27,29,5,5,22,24,33,Test/00003.png
4,60,57,5,5,55,52,11,Test/00004.png


In [ ]:
X_train = []
y_train = []

for _, row in train_df.iterrows():
    img_path = os.path.join(base_path, row["Path"])
    img = image.load_img(img_path, target_size=(32, 32))
    img_array = image.img_to_array(img)

    X_train.append(img_array)
    y_train.append(row["ClassId"])

X_train = np.array(X_train)
y_train = np.array(y_train)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

In [ ]:
X_test = []
y_test = []

for _, row in test_df.iterrows():
    img_path = os.path.join(base_path, row["Path"])
    img = image.load_img(img_path, target_size=(32, 32))
    img_array = image.img_to_array(img)

    X_test.append(img_array)
    y_test.append(row["ClassId"])

X_test = np.array(X_test)
y_test = np.array(y_test)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

In [ ]:
print("Кількість класів:", len(np.unique(y_train)))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random

indices = random.sample(range(len(X_train)), 10)

plt.figure(figsize=(18, 8))

for i, idx in enumerate(indices):
    plt.subplot(2, 5, i + 1)

    img = X_train[idx]
    if img.max() <= 1.0:
        plt.imshow(img)
    else:
        plt.imshow(img.astype("uint8"))

    class_id = np.argmax(y_train[idx]) if len(y_train[idx].shape) > 0 and y_train[idx].shape[0] > 1 else int(y_train[idx])

    plt.title(f"Class {class_id}")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
y_train = to_categorical(y_train, num_classes=43)
y_test = to_categorical(y_test, num_classes=43)

print(f"Форма y_train після перетворення: {y_train.shape}")
print(f"Форма y_test після перетворення: {y_test.shape}")

In [ ]:
height, width, channels = 32, 32, 3
NUM_CATEGORIES = 43

model = Sequential([
    Input(shape=(height, width, channels)),

    Rescaling(1./255),
    RandomFlip("horizontal"),
    RandomRotation(0.05),
    RandomContrast(0.1),

    Conv2D(32, (3,3), padding="same", activation="relu"),
    BatchNormalization(),
    Conv2D(32, (3,3), padding="same", activation="relu"),
    BatchNormalization(),
    MaxPool2D(),

    Conv2D(64, (3,3), padding="same", activation="relu"),
    BatchNormalization(),
    Conv2D(64, (3,3), padding="same", activation="relu"),
    BatchNormalization(),
    MaxPool2D(),

    Conv2D(128, (3,3), padding="same", activation="relu", kernel_regularizer=l2(1e-4)),
    BatchNormalization(),
    MaxPool2D(),

    GlobalAveragePooling2D(),
    Dropout(0.5),
    Dense(128, activation="relu"),
    Dropout(0.3),
    Dense(NUM_CATEGORIES, activation="softmax")
])

In [ ]:
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.metrics import TopKCategoricalAccuracy

LOSS = CategoricalCrossentropy()

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=3e-4),
    loss=LOSS,
    metrics=[
        "accuracy",
        TopKCategoricalAccuracy(k=3, name="top3")
    ]
)

cb = [
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=8, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1)
]

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),  # щоб працював val_accuracy і val_loss
    epochs=50,
    batch_size=32,
    shuffle=True,
    callbacks=cb,
    verbose=1
)

In [ ]:
model.summary()

In [ ]:
training_accuracy = history.history["accuracy"]
training_loss = history.history["loss"]
validation_accuracy = history.history["val_accuracy"]
validation_loss = history.history["val_loss"]

In [ ]:
plt.figure(figsize=(10,5))

plt.subplot(1, 2, 1)
plt.plot(training_accuracy, color = "red", label = "Training Accuracy")
plt.plot(validation_accuracy, color = "cyan", label = "Validation Accuracy")
plt.legend()
plt.grid(True)
plt.title("Accuracy")
plt.xlabel("Number of Epochs")
plt.ylabel("Accuracy")

plt.subplot(1, 2, 2)
plt.plot(training_loss, color = "red", label = "Training Loss")
plt.plot(validation_loss, color = "cyan", label = "Validation Loss")
plt.grid(True)
plt.legend()
plt.title("Loss")
plt.xlabel("Number of epochs")
plt.ylabel("Loss")

plt.tight_layout()
plt.show()

In [ ]:

indices = random.sample(range(len(X_test)), 10)

preds = model.predict(X_test)

sample_images = X_test[indices]
sample_true = y_test[indices]
y_pred = preds[indices]

In [ ]:
plt.figure(figsize=(15, 8))

for i in range(10):
    plt.subplot(2, 5, i + 1)

    img = sample_images[i]

    # FIX
    if img.max() <= 1.0:
        plt.imshow(img)
    else:
        plt.imshow(img.astype("uint8"))

    plt.axis('off')

    true_class = np.argmax(sample_true[i])
    predicted_class = np.argmax(y_pred[i])

    plt.title(f"True: {true_class}\nPred: {predicted_class}")

plt.tight_layout()
plt.show()